### Any project, first inspect the page ###

In [1]:
import requests

url = "https://www.iplt20.com/news"

headers = {
    "User-Agent" : "Mozilla/5.0"
}

responses = requests.get(url, headers = headers)

print(responses.status_code)
print(responses.text[:2000])

200
<!DOCTYPE html><html lang="en" class="bricolage_grotesque_4a227831-module__2b4iXq__variable"><head><meta charSet="utf-8"/><link rel="preconnect" href="/" crossorigin=""/><meta name="viewport" content="width=device-width, initial-scale=1"/><link rel="preload" as="image" imageSrcSet="https://www.iplt20.com/cms/images/jot2qw8s50z1/61QbyC23vBuzs2bM6ruuvT/05a6aab96d8f210ef71480a12bfb217e/Retention_list_announced.png?w=384&amp;q=80&amp;fm=webp 384w, https://www.iplt20.com/cms/images/jot2qw8s50z1/61QbyC23vBuzs2bM6ruuvT/05a6aab96d8f210ef71480a12bfb217e/Retention_list_announced.png?w=720&amp;q=80&amp;fm=webp 720w, https://www.iplt20.com/cms/images/jot2qw8s50z1/61QbyC23vBuzs2bM6ruuvT/05a6aab96d8f210ef71480a12bfb217e/Retention_list_announced.png?w=960&amp;q=82&amp;fm=webp 960w, https://www.iplt20.com/cms/images/jot2qw8s50z1/61QbyC23vBuzs2bM6ruuvT/05a6aab96d8f210ef71480a12bfb217e/Retention_list_announced.png?w=1280&amp;q=82&amp;fm=webp 1280w" imageSizes="(min-width: 1440px) 63.5vw, (min-width:

#### Importing libraries ###

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

### Introducing the agent - a normal browser ###

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [11]:
data = []
pages  = [1, 2, 3] # We will scrap 3 pages

# Visiting each page one by one
for page in pages:
    # Creating page url
    url = f"https://www.iplt20.com/news?page={page}"
    print(f"Scraping: {url}")

    # Downloading page HTML
    response = requests.get(url, headers = headers)

    # Checking page loading status: 200 means successful
    if response.status_code != 200:
        print(f"Failed to load page {page}")
        continue # Skipping the next page

    # Converting HTML code to read, means reading without html tags
    soup = BeautifulSoup(response.text, "html.parser")

    # Finding all news cards
    cards = []
    all_links = soup.find_all("a")

    for link in all_links:
        test_id = link.get("data-testid", "")
        if "all-news-card" in test_id:
            cards.append(link)

    print(f"Cards found: {len(cards)}")

    # Processing each news card
    for card in cards:
        
        # Getting the article url
        article_url = card.get("href", "")
        #print(article_url)
        if article_url.startswith("/"):
            article_url = ( "https://www.iplt20.com" + article_url)
        #print(article_url)

        # Getting the image
        image_tag = card.find("img")
        title = ""
        image_url = ""
        if image_tag: # if image exists
            # Getting image URL
            image_url = image_tag.get("src", "")
            title = image_tag.get("alt", "").strip()

        lines = []

        # Collecting all visible text
        for text in card.stripped_strings:

            lines.append(text)

        news_type = ""
        date = ""

        # First text is usually news type
        if len(lines) >= 1:

            news_type = lines[0]

        # Last text is usually date
        if len(lines) >= 3:

            date = lines[-1]

        # If title was not found from image alt
        # use second line of text
        if title == "" and len(lines) >= 2:

            title = lines[1]
        
        # Creating one record
        row = {
            "News_Type": news_type,
            "Title": title,
            "Date": date,
            "Image_URL": image_url,
            "Article_URL": article_url
        }
        
        data.append(row)

    # Wait 2 seconds before next page
    time.sleep(2)

df = pd.DataFrame(data)
df.head(10)


Scraping: https://www.iplt20.com/news?page=1
Cards found: 16
Scraping: https://www.iplt20.com/news?page=2
Cards found: 16
Scraping: https://www.iplt20.com/news?page=3
Cards found: 16


,News_Type,Title,Date,Image_URL,Article_URL
0,News,Captain Cool to Mentor: The beginning of a new...,4d ago,https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/captain-co...
1,Press Releases,Mohd Shami and M. Siddharth traded to Rajastha...,4d ago,https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/mohd-shami...
2,Press Releases,Harshal Patel traded to Gujarat Titans from Su...,4d ago,https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/harshal-pa...
3,Press Releases,Hardik Pandya and Arshdeep Singh complete majo...,"Wed 11 Nov 2026, 3:45 am",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/hardik-pan...
4,Press Releases,CSK and SRH agree to four-player trade,"Thu 5 Nov 2026, 8:30 am",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/csk-and-sr...
5,Press Releases,T. Natarajan returns to SRH; Zeeshan Ansari mo...,"Wed 14 Oct 2026, 11:28 am",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/t-nataraja...
6,Press Releases,1782201343 Website Thumbnails 67 .pngrishabh-p...,"Tue 23 Jun 2026, 1:15 pm",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/rishabh-pa...
7,Press Releases,Final: RCB vs GT - Code of Conduct Breach,"Mon 1 Jun 2026, 3:00 pm",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/final-rcb-...
8,Match Reports,"TATA IPL 2026, Final: RCB v GT – Match Report","Sun 31 May 2026, 8:44 pm",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/tata-ipl-2...
9,Match Reports,"TATA IPL 2026, Qualifier 2: GT vs RR – Match R...","Fri 29 May 2026, 7:01 pm",https://www.iplt20.com/cms/images/jot2qw8s50z1...,https://www.iplt20.com/news/article/tata-ipl-2...
